# Select final aging genes from one trained fold

Run this notebook after a **full pruning run** in the repository README. One-epoch Quick run and eight-epoch checks are insufficient.

Install `requirements-gene-selection.txt` in the model environment, then start `python -s -m jupyterlab` from the repository root. Select that environment's Python kernel and run all cells in order.

Inputs are the README's prepared HDF5 files, saved feature masks and `data/header.txt`. Selection uses one configurable fold, not a five-fold consensus. Gene names retain their original case and column order.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
REPO = next((p for p in (start, *start.parents)
             if (p / "run_cross_validation.py").is_file()
             and (p / "model" / "gene_selection.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Start Jupyter inside your cloned sAge repository.")
sys.path.insert(0, str(REPO / "model"))
from gene_selection import run_selection, AGE_MAPPING


## 1. Configure the run

Defaults match `prepared_data/Heart` and `outputs/Heart` in the README. Choose the fold used for your analysis; `fold_0` is an example, not a requirement. Use a new `OUTPUT_DIR` for a repeat.

`SELECTION_SPLIT = "initial_test"` preserves the previous notebook's initial training/test split for regression and elbow selection. **The test data participate in selection; these metrics are not independent final-test performance.** Set `"validation"` to fit on the chosen fold's training data and select on its validation data, leaving `initial_split/test.h5` unused by this notebook. This changes the analysis and may change the selected genes; record the setting used for manuscript results.

`MAX_FEATURES = 1011` preserves the previous notebook's candidate-count cutoff (not a column index). Set `None` to evaluate all saved counts; large regressions may take considerably longer. At least six candidate counts and a non-flat MAE curve are required.

The original mouse age mapping in months is `{0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}`. Other datasets need their own mapping and an ordered header matching the expression columns.


In [ ]:
DATA_DIR = Path(os.environ.get("SAGE_GENE_SELECTION_DATA_DIR", str(REPO / "prepared_data" / "Heart")))
RUN_DIR = Path(os.environ.get("SAGE_GENE_SELECTION_RUN_DIR", str(REPO / "outputs" / "Heart")))
HEADER_FILE = REPO / "data" / "header.txt"
FOLD = os.environ.get("SAGE_GENE_SELECTION_FOLD", "fold_0")
SELECTION_SPLIT = os.environ.get("SAGE_GENE_SELECTION_SPLIT", "initial_test")
MAX_FEATURES = 1011
OUTPUT_DIR = Path(os.environ.get("SAGE_GENE_SELECTION_OUTPUT_DIR", str(RUN_DIR / FOLD / "gene_selection")))
AGE_MAPPING_MONTHS = dict(AGE_MAPPING)

print("Dataset:", DATA_DIR)
print("Training run:", RUN_DIR, "| Fold:", FOLD)
print("Selection split:", SELECTION_SPLIT)
print("Output:", OUTPUT_DIR)


## 2. Evaluate masks and choose the elbow

Fit an ordinary least-squares age regressor for each saved feature mask and evaluate PCC, MAE and RMSE on the configured selection set. As before, regression uses the stored expression without adding the classifier's normalization.

The MAE elbow retains the MAD-filtered, two-segment search up to the minimum-MAE region. Its objective is corrected to **sum of squared residuals**. The old code squared the sum of residuals, which can cancel to near zero for a fitted linear model. The corrected implementation is therefore not guaranteed to reproduce previously exported gene lists.

Missing data, incomplete training, header/mask length mismatches, invalid masks, insufficient candidate counts and flat MAE curves raise explicit errors.


In [ ]:
rows, summary, figure = run_selection(
    DATA_DIR, RUN_DIR, HEADER_FILE, FOLD, OUTPUT_DIR,
    selection_split=SELECTION_SPLIT,
    max_features=MAX_FEATURES,
    age_mapping=AGE_MAPPING_MONTHS,
)


## 3. Inspect and use the results

The plot shows selection-set metrics, not independent test estimates. Genes are selected for this dataset and fold; this does not establish biological causality or reproduce every manuscript result.

Files under `OUTPUT_DIR`:

- `final_aging_genes.txt`: one gene per line.
- `final_aging_genes.csv`: genes with zero-based original feature indices.
- `selected_feature_mask.txt`: binary mask in the original column order.
- `feature_metrics.csv`: all evaluated feature counts and regression metrics.
- `elbow_metrics.png` and `elbow_metrics.pdf`: selection curves.
- `selection_summary.json`: fold, split, settings, selected checkpoint, checksums and training command.

The supplied header is mapped by position. Matching dimensions alone cannot prove biological gene-column identity; keep the original feature order throughout preprocessing, training and selection.


In [ ]:
import pandas as pd
from IPython.display import display

print(f"Selected {summary['selected']['Gene_Count']} genes from {summary['fold']}.")
display(pd.DataFrame(rows).drop(columns="Mask_Path").round(4).head(10))
display(pd.read_csv(OUTPUT_DIR / "final_aging_genes.csv").head(20))
print("Full gene list:", OUTPUT_DIR / "final_aging_genes.txt")
print("Run record:", OUTPUT_DIR / "selection_summary.json")
